# GRDP per Capita — Silver
Snapshot upsert | Whole pesos per person | Stable existing IDs.

Run all cells in one session. Only one writer may use this target: maximum concurrent job runs = 1; avoid overlapping manual writes.

Scope defaults to 2021–2025. Bronze must contain the complete selected-year snapshot. Missing existing keys fail the load for review; no automatic deletes.

Existing names, numeric price_basis and region mapping are preserved. A verified shared geography crosswalk and boundary compatibility (especially NIR) are still required before Gold joins.

Four lineage columns are added automatically to existing tables. First run backfills lineage. Unchanged Bronze reruns write zero rows; changed ingestion metadata updates lineage while retaining IDs.

Materializing on the driver is appropriate for this small indicator dataset. This notebook has not been executed against Databricks.


In [ ]:
%python
from datetime import datetime, timezone
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType

SOURCE = "pondolens.dev_bronze.grdp_per_capita"
TARGET = "pondolens.dev_silver.grdp_per_capita"
START_YEAR, END_YEAR = 2021, 2025
KEY = ["year", "region_code", "price_basis", "geographic_level"]
BASE_ID, MAX_ID = 100000000, 2147483647
RUN_ID = "silver_grdp_pc_" + uuid.uuid4().hex
RUN_TS = datetime.now(timezone.utc)
assert START_YEAR <= END_YEAR

bronze = spark.table(SOURCE)
required = {"region_id", "region", "year", "GRDP_per_capita_amount",
            "price_basis", "geographic_level", "_rescued_data",
            "_drive_file_id", "_source_file", "_run_id", "_content_hash"}
if required - set(bronze.columns):
    raise ValueError(f"Missing Bronze columns: {sorted(required - set(bronze.columns))}")
# Materialize this small indicator snapshot so checks and writes use identical data.
snapshot = spark.createDataFrame(bronze.collect(), bronze.schema)
snapshot.createOrReplaceTempView("grdp_pc_snapshot")
bad = spark.sql("""
SELECT * FROM grdp_pc_snapshot
WHERE TRY_CAST(year AS INT) IS NULL
   OR NULLIF(TRIM(_rescued_data), '') IS NOT NULL
""")
if bad.limit(1).count():
    display(bad)
    raise ValueError("Invalid years or rescued records; repair Bronze first.")

stage = spark.sql(f"""
WITH normalized AS (
 SELECT NULLIF(UPPER(TRIM(region_id)), '') AS source_region_code,
        NULLIF(TRIM(REGEXP_REPLACE(region, r'\\s+', ' ')), '') AS region_name,
        TRY_CAST(year AS INT) AS year,
        TRY_CAST(ROUND(TRY_CAST(GRDP_per_capita_amount AS DECIMAL(30,15)),0) AS BIGINT)
          AS grdp_per_capita,
        LOWER(TRIM(price_basis)) AS source_price_basis,
        NULLIF(LOWER(TRIM(geographic_level)), '') AS source_geographic_level,
        _drive_file_id AS source_drive_file_id, _source_file AS source_file,
        _run_id AS source_ingestion_run_id, _content_hash AS source_content_hash
 FROM grdp_pc_snapshot
 WHERE TRY_CAST(year AS INT) BETWEEN {START_YEAR} AND {END_YEAR}
)
SELECT CASE WHEN UPPER(region_name) = 'PHILIPPINES' THEN 'PHL'
            WHEN UPPER(region_name) = 'MIMAROPA REGION' THEN 'IV-B'
            ELSE source_region_code END AS region_code,
       region_name, year, grdp_per_capita,
       CASE WHEN source_price_basis = 'at current prices' THEN year
            WHEN source_price_basis = 'at constant 2018 prices' THEN 2018 END AS price_basis,
       CASE WHEN UPPER(region_name) = 'PHILIPPINES' THEN 'national'
            ELSE source_geographic_level END AS geographic_level,
       source_drive_file_id, source_file, source_ingestion_run_id, source_content_hash
FROM normalized
""")
incoming = stage.collect()
display(stage.groupBy("year", "geographic_level", "price_basis").count())

def validate_rows(rows, existing=False):
    if not rows and not existing:
        raise ValueError("No incoming data in selected scope.")
    keys, ids = set(), set()
    for raw in rows:
        r = raw.asDict() if hasattr(raw, "asDict") else raw
        key = tuple(r.get(c) for c in KEY)
        if any(v is None or (isinstance(v, str) and not v.strip()) for v in key):
            raise ValueError(f"Missing business key: {key}")
        if key in keys:
            raise ValueError(f"Duplicate business key: {key}")
        keys.add(key)
        if not r.get("region_name") or not r["region_name"].strip() or r.get("grdp_per_capita") is None or r["grdp_per_capita"] <= 0:
            raise ValueError(f"Invalid region name or amount: {key}")
        if r["price_basis"] not in (2018, r["year"]):
            raise ValueError(f"Unknown price basis: {key}")
        if r["geographic_level"] not in ("region", "national"):
            raise ValueError(f"Unknown geographic level: {key}")
        national = r["geographic_level"] == "national"
        if national != (r["region_code"] == "PHL") or national != (r["region_name"].upper() == "PHILIPPINES"):
            raise ValueError(f"Inconsistent national geography: {key}")
        if existing:
            value = r.get("grdp_per_capita_id")
            if value is None or not BASE_ID < value <= MAX_ID or value in ids:
                raise ValueError(f"Invalid or duplicate existing ID: {value}")
            ids.add(value)
        else:
            for c in ("source_drive_file_id", "source_file", "source_ingestion_run_id", "source_content_hash"):
                if not r.get(c) or not r[c].strip():
                    raise ValueError(f"Missing {c}: {key}")
    return keys

incoming_keys = validate_rows(incoming)
target_rows = spark.table(TARGET).collect() if spark.catalog.tableExists(TARGET) else []
existing_keys = validate_rows(target_rows, existing=True)
removed = [r for r in target_rows if START_YEAR <= r["year"] <= END_YEAR
           and tuple(r[c] for c in KEY) not in incoming_keys]
if removed:
    display(spark.createDataFrame(removed))
    raise ValueError("Existing scoped keys are absent from Bronze. Review source removal or key correction first.")
last_id = max([BASE_ID] + [r["grdp_per_capita_id"] for r in target_rows])
new_keys = sorted(incoming_keys - existing_keys)
if last_id + len(new_keys) > MAX_ID:
    raise ValueError("New IDs exceed INT capacity; migrate to BIGINT first.")
new_ids = {key: last_id + i for i, key in enumerate(new_keys, 1)}
print(f"Validated: {len(incoming)} source records; {len(new_keys)} new keys.")

# Only target schema creation/migration follows successful data validation.
spark.sql("CREATE SCHEMA IF NOT EXISTS pondolens.dev_silver")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TARGET} (
 grdp_per_capita_id INT, region_code STRING, region_name STRING, year INT,
 grdp_per_capita BIGINT, price_basis INT, geographic_level STRING,
 processing_run_id STRING, processed_at TIMESTAMP,
 source_drive_file_id STRING, source_file STRING,
 source_ingestion_run_id STRING, source_content_hash STRING
) USING DELTA
""")
lineage = ["source_drive_file_id", "source_file", "source_ingestion_run_id", "source_content_hash"]
missing_lineage = [c for c in lineage if c not in spark.table(TARGET).columns]
if missing_lineage:
    spark.sql(f"ALTER TABLE {TARGET} ADD COLUMNS (" +
              ", ".join(f"{c} STRING" for c in missing_lineage) + ")")
# Allocate once; MERGE cannot lazily reevaluate MAX(ID).
schema = StructType(stage.schema.fields + [StructField("new_id", IntegerType(), True)])
rows = [tuple(r[c] for c in stage.columns) + (new_ids.get(tuple(r[c] for c in KEY)),) for r in incoming]
merge_source = spark.createDataFrame(rows, schema)
merge_source = merge_source.withColumn("processing_run_id", F.lit(RUN_ID)).withColumn("processed_at", F.lit(RUN_TS))
merge_source.createOrReplaceTempView("grdp_pc_merge_source")
on = " AND ".join(f"t.{c} = s.{c}" for c in KEY)
changed = ["region_name", "grdp_per_capita"] + lineage
condition = " OR ".join(f"NOT (t.{c} <=> s.{c})" for c in changed)
updates = ", ".join(f"t.{c} = s.{c}" for c in changed + ["processing_run_id", "processed_at"])
columns = ["grdp_per_capita_id"] + stage.columns + ["processing_run_id", "processed_at"]
values = ["s.new_id"] + [f"s.{c}" for c in stage.columns + ["processing_run_id", "processed_at"]]
spark.sql(f"""
MERGE INTO {TARGET} t USING grdp_pc_merge_source s ON {on}
WHEN MATCHED AND ({condition}) THEN UPDATE SET {updates}
WHEN NOT MATCHED THEN INSERT ({", ".join(columns)}) VALUES ({", ".join(values)})
""")

actual = spark.table(TARGET)
actual_rows = actual.collect()
validate_rows(actual_rows, existing=True)
scoped = actual.where(F.col("year").between(START_YEAR, END_YEAR)).select(*stage.columns)
expected = spark.createDataFrame(incoming, stage.schema)
diff = expected.exceptAll(scoped).unionByName(scoped.exceptAll(expected))
if diff.limit(1).count():
    display(diff)
    raise ValueError("Post-load reconciliation failed.")
saved_ids = {tuple(r[c] for c in KEY): r["grdp_per_capita_id"] for r in actual_rows}
if any(saved_ids[tuple(r[c] for c in KEY)] != r["grdp_per_capita_id"] for r in target_rows):
    raise ValueError("Existing IDs changed.")
display(actual.groupBy("year").agg(
    F.count("*").alias("records"),
    F.sum(F.when(F.col("processing_run_id") == RUN_ID, 1).otherwise(0)).alias("written_this_run")))
print("PASS: scoped source matches Silver; existing IDs retained.")
